# patgen — mining templates from messy bilingual SMS

This notebook walks the whole approach on a synthetic corpus that mimics real
banking traffic: Arabic + English, Arabic-Indic digits, tashkeel, bidi marks,
cp1252 mojibake in the middle of otherwise clean text, and truncated tails.

1. the data and what is wrong with it
2. normalization + mojibake repair
3. tokenization + entity masking (the step that makes Drain work here)
4. Drain-style clustering and wildcard refinement
5. the learned library: templates, typed slots, coverage
6. production matching + throughput
7. what stays unmatched, and tuning

In [1]:
import random
import re
import sys
import time
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from patgen import (  # noqa: E402
    DrainTree,
    LearnConfig,
    TemplateMatcher,
    learn_templates,
    normalize,
    prepare,
    repair_mojibake,
    tokenize,
)
from patgen.entities import mask_tokens  # noqa: E402
from patgen.io_csv import read_texts  # noqa: E402
from patgen.report import coverage_report  # noqa: E402

CSV = Path.cwd().parent / "examples" / "sms_sample.csv"
messages = list(read_texts([CSV]))
len(messages), messages[0]

(5000,
 'Bill payment AED 35,537.14 to jarir bookstore succeeded. Fee AED 82.31. Ref 565623510')

## 1. What the raw data looks like

In [2]:
random.seed(7)
for m in random.sample(messages, 8):
    print(repr(m))

'\u200fSalary of SAR 49,325.80 credited to account ***696 on 24/04/2024\u200e'
'Transfer of AED ٨٢,٣٩٠.٨٣ to acme trading llc completed. Ref ٤٥٥٦١٨٦٧٤. Balance AED ٣٤٠,١٩٨.٥٦'
'Your OTP is 417572. Valid for 10 minutes. Do not share '
'Your OTP is 628969. Valid for 5 minutes. Do not share it with anyone'
'Your OTP is 393367. Valid for 15 minutes. Do not share it with anyone'
'عملية شراء بمبلغ 67,313.14 USD لدى amazon.sa بالبطاقة xxxx8599 بتاريخ 19/10/2024 الرصيد المتاح 135,229.48'
'رمز Ø§Ù„ØªØ\xadÙ‚Ù‚ Ø§Ù„Ø®Ø§Øµ Ø¨Ùƒ Ù‡Ùˆ 641805 ØµØ§Ù„Ø\xad لمدة 10 دقائق لا تشاركه مع احد'
'Salary of SAR 24,022.32 credited to account ***834 on 22/05/2024 -الأهلي'


Three separate problems in one column:

* **two scripts** — Arabic and English templates, often mixed inside a message;
* **encoding damage** — `Ø±.Ø³` is `ر.س` that went through UTF-8 → cp1252,
  and it is usually only *part* of the message, so a whole-string round trip
  cannot fix it;
* **noise** — Arabic-Indic digits, tashkeel, bidi marks, double spaces,
  truncated tails.

In [3]:
broken = [m for m in messages if re.search("[ÂÃØÙÚÛ][\u0080-\u00ff\u0152-\u0178]", m)]
print(f"{len(broken)}/{len(messages)} messages carry mojibake\n")
for m in broken[:3]:
    print("raw :", m)
    print("fixed:", repair_mojibake(m), "\n")

273/5000 messages carry mojibake

raw : Bill payment Ø±.Ø³ 37,955.74 to Ù‡Ù†Ù‚Ø±Ø³ØªÙŠØ´Ù† succeeded. Fee ر.س 8.68. Ref 173833652
fixed: Bill payment ر.س 37,955.74 to هنقرستيشن succeeded. Fee ر.س 8.68. Ref 173833652 

raw : تم Ø§ÙŠØ¯Ø§Ø¹ Ø±Ø§ØªØ¨ Ø¨Ù…Ø¨Ù„Øº 25,119.21 SAR Ùي الحساب ***920 بتاريخ 2024-04-29
fixed: تم ايداع راتب بمبلغ 25,119.21 SAR  ي الحساب ***920 بتاريخ 2024-04-29 

raw : عزيزنا Ø§Ù„Ø¹Ù…ÙŠÙ„ ØªÙ… ØªÙØ¹ÙŠÙ„ Ø®Ø¯Ù…Ø© apple pay Ø¹Ù„Ù‰ Ø­Ø³Ø§Ø¨ك ***735 للاستفسار اتصل على 966585890364
fixed: عزيزنا العميل تم Ø¹Ù خدمة apple pay على حسابك ***735 للاستفسار اتصل على 966585890364 



## 2. Normalization

In [4]:
samples = [
    "‏سحب نقدي ٦٩,٩٥١.٧٠ ر.س من الحساب ***815‎",
    "Bill payment Ø±.Ø³ 37,955.74 to ACME",
    "رَصيدُك الحالي ٢٥٠٫٥٠ ريال",
    "عملية شراء لدى صيدلية النهدي",
]
for s in samples:
    print(f"{s!r}\n  -> {normalize(s)!r}")

'\u200fسحب نقدي ٦٩,٩٥١.٧٠ ر.س من الحساب ***815\u200e'
  -> 'سحب نقدي 69,951.70 ر.س من الحساب ***815'
'Bill payment Ø±.Ø³ 37,955.74 to ACME'
  -> 'bill payment ر.س 37,955.74 to acme'
'رَصيدُك الحالي ٢٥٠٫٥٠ ريال'
  -> 'رصيدك الحالي 250.50 ريال'
'عملية شراء لدى صيدلية النهدي'
  -> 'عمليه شراء لدي صيدليه النهدي'


Folding (`أإآ→ا`, `ة→ه`, `ى→ي`), digit conversion and mojibake repair all
happen *before* clustering, so the same sentence written four different ways
collapses onto one template instead of four.

## 3. Tokenization + entity masking

In [5]:
msg = "Purchase of ر.س 4,467.79 at starbucks on card ending ****5744 on 29/07/2024 02:48. Available balance ر.س 110,579.86"
prepared = prepare(msg)
print("canonical:", prepared.canonical, "\n")
for m in mask_tokens(tokenize(prepared.canonical)):
    flag = "  <-- entity" if m.is_entity else ""
    print(f"{m.key:<12} {m.value}{flag}")

canonical: purchase of ر . س 4,467.79 at starbucks on card ending * * * * 5744 on 29/07/2024 02:48 . available balance ر . س 110,579.86 

purchase     purchase
of           of
<CURRENCY>   ر . س  <-- entity
<AMOUNT>     4,467.79  <-- entity
at           at
starbucks    starbucks
on           on
card         card
ending       ending
<CARD>       * * * * 5744  <-- entity
on           on
<DATETIME>   29/07/2024 02:48  <-- entity
.            .
available    available
balance      balance
<CURRENCY>   ر . س  <-- entity
<AMOUNT>     110,579.86  <-- entity


This is the key difference from vanilla Drain3. Values are replaced by
**typed** placeholders before clustering, so amounts, dates and card tails
never split a cluster, and the placeholder type is carried into the template
(a plain `<*>` would lose it).

## 4. Clustering, and why wildcards get refined

In [6]:
tree = DrainTree()
for m in messages[:2000]:
    tree.add(prepare(m).keys)
print(f"{len(tree.clusters)} raw clusters from 2000 messages")
for c in sorted(tree.clusters, key=lambda c: -c.count)[:6]:
    print(f"{c.count:>5}  {' '.join(c.tokens)}")

122 raw clusters from 2000 messages
  137  your otp is <NUM> . valid for <NUM> minutes . do not share it with anyone <*>
  133  atm withdrawal <CURRENCY> <AMOUNT> from account <CARD> on <DATE> . available <*>
  117  رمز التحقق الخاص بك <*> <NUM> صالح لمده <NUM> دقايق لا تشاركه مع احد
  117  خصم رسوم <AMOUNT> <CURRENCY> علي الحساب <CARD> رقم <*>
  106  salary of <CURRENCY> <AMOUNT> credited to account <CARD> <*>
   72  dear customer , service <*>


A single truncated message widens a cluster into `<*>`, swallowing fields
that were perfectly extractable. The learner replays every wildcard against the
messages that filled it and splices back the dominant filling — or the dominant
prefix/suffix around the part that genuinely varies — marking it optional when
some messages left it empty.

In [7]:
t0 = time.perf_counter()
library = learn_templates(messages)
print(f"{len(messages)} messages -> {len(library.templates)} templates in {time.perf_counter()-t0:.2f}s")
for t in library.templates[:10]:
    print(f"{t.count:>5}  {t.text}")

5000 messages -> 74 templates in 0.64s
  332  transfer of <CURRENCY:currency> <AMOUNT:amount> to <TEXT:beneficiary>? completed . ref <NUM:ref> . balance <CURRENCY:currency_2> <AMOUNT:balance>
  328  purchase of <CURRENCY:currency> <AMOUNT:amount> at <TEXT:merchant> on card ending <CARD:card> on <DATETIME:date> . available balance <CURRENCY:currency_2> <AMOUNT:balance>
  324  عمليه شراء بمبلغ <AMOUNT:amount> <CURRENCY:currency> لدي <TEXT:merchant>? بالبطاقه <CARD:card> بتاريخ <DATE:date> الرصيد المتاح <AMOUNT:balance>
  320  خصم رسوم <AMOUNT:fee> <CURRENCY:currency> علي الحساب <CARD:account> رقم العمليه <NUM:ref>
  315  your otp is <NUM:otp> . valid for <NUM:num> minutes . do not share it with anyone thank <TEXT:text>?
  313  dear customer , service <TEXT:service> was activated on account <CARD:account> . call <PHONE:phone> for help
  300  عزيزنا العميل تم تفعيل خدمه <TEXT:service>? علي حسابك <CARD:card> للاستفسار اتصل علي <PHONE:phone>
  299  سحب نقدي <AMOUNT:amount> <CURRENCY:currency

## 5. Typed slots per template

In [8]:
t = library.templates[1]
print(t.text, "\n")
for slot in t.slots:
    print(f"{slot.key:<12} {slot.entity:<9} distinct={slot.cardinality:<5} e.g. {slot.examples[:3]}")

purchase of <CURRENCY:currency> <AMOUNT:amount> at <TEXT:merchant> on card ending <CARD:card> on <DATETIME:date> . available balance <CURRENCY:currency_2> <AMOUNT:balance> 

currency     CURRENCY  distinct=4     e.g. ['ر . س', 'usd', 'sar']
amount       AMOUNT    distinct=319   e.g. ['13,198.50', '46,571.46', '84,431.64']
merchant     TEXT      distinct=9     e.g. ['jarir bookstore', 'starbucks', 'بنده']
card         CARD      distinct=313   e.g. ['* * * * 1287', 'xxxx 7041', '* * * * 7992']
date         DATETIME  distinct=319   e.g. ['2024-03-15 17:11', '2024-05-19 22:27', '10/07/2024 20:15']
currency_2   CURRENCY  distinct=4     e.g. ['ر . س', 'usd', 'sar']
balance      AMOUNT    distinct=319   e.g. ['271,609.05', '147,762.28', '484,607.95']


In [9]:
counts = Counter(slot.key for tpl in library.templates for slot in tpl.slots)
counts.most_common(15)

[('currency', 45),
 ('amount', 43),
 ('card', 24),
 ('account', 21),
 ('ref', 20),
 ('date', 19),
 ('currency_2', 15),
 ('balance', 15),
 ('text', 15),
 ('phone', 15),
 ('beneficiary', 14),
 ('fee', 10),
 ('merchant', 9),
 ('service', 9),
 ('amount_2', 7)]

## 6. Production matching

In [10]:
matcher = TemplateMatcher(library)
for m in random.sample(messages, 5):
    r = matcher.match(m)
    print(m)
    print("   ->", r.template_id, r.entities if r else None, "\n")

ATM withdrawal SAR ٧٩,٦٦٤.٧٢ from account ***٢١٣ on ٢٠٢٤-٠٢-٠٤. Available balance SAR ٦٨,٣٩٩.١٨
   -> t00024 {'currency': 'sar', 'amount': '79,664.72', 'account': '***213', 'date': '2024-02-04', 'currency_2': 'sar', 'balance': '68,399.18'} 

رمز التحقق الخاص بك هو 174019 صالح لمدة 15 دقائق لا تشاركه مع احد
   -> t00008 {'otp': '174019', 'num': '15'} 

سحب نقدي 50,180.71 AED من الحساب ***253 بتاريخ 2024-03-12 الرصيد المتاح 363,516.17
   -> t00002 {'amount': '50,180.71', 'currency': 'aed', 'account': '***253', 'date': '2024-03-12', 'balance': '363,516.17'} 

Purchase of USD ٤٢,٥٩٧.٥٨ at starbucks on card ending ****٧٦٨٣ on ٢٠٢٤-٠٨-٠٦ ١٥:٤٨. Available balance USD ١٦٨,٤٨٨.٥٥
   -> t00011 {'currency': 'usd', 'amount': '42,597.58', 'merchant': 'starbucks', 'card': '****7683', 'date': '2024-08-06 15:48', 'currency_2': 'usd', 'balance': '168,488.55'} 

Your OTP is 351688. Valid for 10 minutes. Do not share it with anyone
   -> t00066 {'otp': '351688', 'num': '10', 'text': 'it with anyone'} 



In [11]:
batch = messages * 3
t0 = time.perf_counter()
hits = sum(1 for r in matcher.match_many(batch) if r)
elapsed = time.perf_counter() - t0
print(f"{len(batch)} messages in {elapsed:.2f}s = {len(batch)/elapsed:,.0f} msg/s, {hits/len(batch):.1%} matched")

15000 messages in 1.52s = 9,891 msg/s, 95.0% matched


One normalization pass, then a bucket lookup on the first literal token and
a handful of alternation regexes with named groups — the inner loop runs in the
C regex engine, not in Python.

## 7. What is left unmatched, and tuning

In [12]:
stats = coverage_report(library, messages)
print(f"coverage {stats['coverage']:.1%}")
for m in stats["unmatched_samples"][:8]:
    print(" ", m)

coverage 95.0%
  تم Ø§ÙŠØ¯Ø§Ø¹ Ø±Ø§ØªØ¨ Ø¨Ù…Ø¨Ù„Øº 25,119.21 SAR Ùي الحساب ***920 بتاريخ 2024-04-29
  عزيزنا العميل تم تفعيل خدمة المدفوعات الدولية على حسابك ***390 للاستفسار ات
  عزيزنا Ø§Ù„Ø¹Ù…ÙŠÙ„ ØªÙ… ØªÙØ¹ÙŠÙ„ Ø®Ø¯Ù…Ø© apple pay Ø¹Ù„Ù‰ Ø­Ø³Ø§Ø¨ك ***735 للاستفسار اتصل على 966585890364
  Your  card ****8735 was declined at صيدلية النهدي due to ins
  تم ايداع راتب بمبلغ 5,023.82 ر.س في الحساب ***582 بتا
  تم ايداع راتب بمبلغ 75,578.05 USD في الحساب ***571 بتا
  خصم رسوم 36.69 AED على الحساب ***411 رقم العمل
  تم Ø§ÙŠØ¯Ø§Ø¹ Ø±Ø§ØªØ¨ Ø¨Ù…Ø¨Ù„Øº 31,541.93 AED Ùي الحساب ***615 بتاريخ 2024-05-12


The residue is genuinely damaged traffic: messages cut mid-word by the SMS
gateway, or mojibake where one byte of the UTF-8 pair was dropped and nothing
can decode it back. Those are the rows worth alerting on, not templating.

In [13]:
for sim in (0.3, 0.4, 0.5, 0.6, 0.7):
    lib = learn_templates(messages, LearnConfig(sim_threshold=sim))
    cov = coverage_report(lib, messages)["coverage"]
    print(f"sim={sim:<4} templates={len(lib.templates):<5} coverage={cov:.1%}")

sim=0.3  templates=75    coverage=94.6%


sim=0.4  templates=67    coverage=95.3%


sim=0.5  templates=74    coverage=95.0%


sim=0.6  templates=75    coverage=93.7%


sim=0.7  templates=73    coverage=93.0%


Higher similarity = more, tighter templates; lower = fewer, more generic
ones. `--min-support` then trims the long tail before the model is saved with
`library.save("model.json")` and loaded by the serving process.